# 🏆 2026 FIFA World Cup Predictor: Feature Engineering Pipeline
This notebook processes raw international football match data into a machine learning-ready feature matrix.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore') # Keeps the notebook clean from pandas slice warnings

## Step 1: Base Cleaning & Target Configuration
This function handles formatting dates, defining the win/draw/loss target, merging penalty shootout data, and engineering tournament weights and host advantages.

In [3]:
def load_and_clean_data(results_path, shootouts_path):
    # Load datasets
    df = pd.read_csv(results_path)
    shootouts = pd.read_csv(shootouts_path)
    
    # Format dates
    df['date'] = pd.to_datetime(df['date'])
    shootouts['date'] = pd.to_datetime(shootouts['date'])
    
    # Sort chronologically
    df = df.sort_values('date').reset_index(drop=True)
    
    # Target Calculations & Goal Difference
    df['goal_difference'] = df['home_score'] - df['away_score']
    
    conditions = [
        (df['home_score'] > df['away_score']),
        (df['home_score'] == df['away_score']),
        (df['home_score'] < df['away_score'])
    ]
    df['outcome'] = np.select(conditions, [2, 1, 0])
    
    # Incorporate penalty shootout results
    df = df.merge(shootouts[['date', 'home_team', 'away_team', 'winner']], 
                  on=['date', 'home_team', 'away_team'], how='left')
    df.loc[(df['outcome'] == 1) & (df['winner'] == df['home_team']), 'outcome'] = 2
    df.loc[(df['outcome'] == 1) & (df['winner'] == df['away_team']), 'outcome'] = 0
    df.drop(columns=['winner'], inplace=True)
    
    # Match Context & Tournament Weights
    tournament_weights = {
        'Friendly': 1, 'FIFA World Cup qualification': 2, 'UEFA Euro qualification': 2,
        'Copa América qualification': 2, 'African Cup of Nations qualification': 2,
        'CONCACAF Nations League': 2, 'UEFA Nations League': 2, 'AFC Asian Cup': 3,
        'African Cup of Nations': 3, 'Copa América': 3, 'UEFA Euro': 3,
        'CONCACAF Gold Cup': 3, 'FIFA World Cup': 4
    }
    df['match_importance'] = df['tournament'].map(tournament_weights).fillna(1)
    
    # 2026 Host Advantage Override
    hosts_2026 = ['United States', 'Mexico', 'Canada']
    df.loc[(df['home_team'].isin(hosts_2026)) & (df['country'] == df['home_team']), 'neutral'] = False
    df.loc[(df['away_team'].isin(hosts_2026)) & (df['country'] == df['away_team']), 'neutral'] = False
    df['home_has_advantage'] = np.where(df['neutral'] == False, 1, 0)
    
    return df

## Step 2: Team Rolling Features
Calculates rolling momentum variables chronologically (win rates, goals scored/conceded, and days of rest).

In [4]:
def build_team_rolling_features(df, window=5):
    home_df = df[['date', 'home_team', 'home_score', 'away_score', 'outcome']].copy()
    home_df.columns = ['date', 'team', 'goals_scored', 'goals_conceded', 'match_result']
    home_df['is_win'] = (home_df['match_result'] == 2).astype(int)
    
    away_df = df[['date', 'away_team', 'away_score', 'home_score', 'outcome']].copy()
    away_df.columns = ['date', 'team', 'goals_scored', 'goals_conceded', 'match_result']
    away_df['is_win'] = (away_df['match_result'] == 0).astype(int)
    
    team_timeline = pd.concat([home_df, away_df]).sort_values(['team', 'date']).reset_index(drop=True)
    
    team_timeline['days_rest'] = team_timeline.groupby('team')['date'].diff().dt.days.fillna(30)
    team_timeline['days_rest'] = team_timeline['days_rest'].clip(upper=30)
    
    grouped = team_timeline.groupby('team')
    
    team_timeline['win_rate'] = grouped['is_win'].shift(1).rolling(window, min_periods=1).mean().reset_index(0, drop=True)
    team_timeline['avg_goals_scored'] = grouped['goals_scored'].shift(1).rolling(window, min_periods=1).mean().reset_index(0, drop=True)
    team_timeline['avg_goals_conceded'] = grouped['goals_conceded'].shift(1).rolling(window, min_periods=1).mean().reset_index(0, drop=True)
    
    team_timeline['win_rate'] = team_timeline['win_rate'].fillna(0.33)
    team_timeline['avg_goals_scored'] = team_timeline['avg_goals_scored'].fillna(1.0)
    team_timeline['avg_goals_conceded'] = team_timeline['avg_goals_conceded'].fillna(1.0)
    
    home_features = team_timeline.add_prefix('home_').rename(columns={'home_date': 'date', 'home_team': 'home_team'})
    away_features = team_timeline.add_prefix('away_').rename(columns={'away_date': 'date', 'away_team': 'away_team'})
    
    df = df.merge(home_features[['date', 'home_team', 'home_days_rest', 'home_win_rate', 'home_avg_goals_scored', 'home_avg_goals_conceded']], on=['date', 'home_team'], how='left')
    df = df.merge(away_features[['date', 'away_team', 'away_days_rest', 'away_win_rate', 'away_avg_goals_scored', 'away_avg_goals_conceded']], on=['date', 'away_team'], how='left')
    
    return df

## Step 3: Head-to-Head Features
Computes historical dominance and rivalry tracking between specific pairings.

In [5]:
def build_head_to_head_features(df):
    df['pair'] = df.apply(lambda row: tuple(sorted([row['home_team'], row['away_team']])), axis=1)
    
    df['h2h_home_wins'] = 0
    df['h2h_total_matches'] = 0
    df['h2h_cumulative_goal_diff'] = 0.0
    
    pair_history = {}
    
    for idx, row in df.iterrows():
        pair = row['pair']
        home = row['home_team']
        gd = row['goal_difference']
        outcome = row['outcome']
        
        if pair not in pair_history:
            pair_history[pair] = {'total': 0, 'home_wins_count': 0, 'cum_gd': 0.0}
            
        stats = pair_history[pair]
        
        if stats['total'] > 0:
            df.at[idx, 'h2h_total_matches'] = stats['total']
            df.at[idx, 'h2h_home_win_percentage'] = stats['home_wins_count'] / stats['total']
            df.at[idx, 'h2h_goal_diff_historical'] = stats['cum_gd']
        else:
            df.at[idx, 'h2h_total_matches'] = 0
            df.at[idx, 'h2h_home_win_percentage'] = 0.5
            df.at[idx, 'h2h_goal_diff_historical'] = 0.0
            
        stats['total'] += 1
        stats['cum_gd'] += gd if home == row['home_team'] else -gd
        if outcome == 2:
            stats['home_wins_count'] += 1
            
    df.drop(columns=['pair', 'h2h_home_wins'], inplace=True)
    return df

## Step 4: Execute Pipeline
Run the pipeline to generate the final feature matrix (`X`) and target variable (`y`).

In [7]:
# 1. Load data 
# Make sure 'results.csv' and 'shootouts.csv' are in the same folder as this notebook
base_df = load_and_clean_data('./data/results.csv', './data/shootouts.csv')

# 2. Build rolling features (Using a 5-match window)
model_ready_df = build_team_rolling_features(base_df, window=5)

# 3. Build head-to-head features
model_ready_df = build_head_to_head_features(model_ready_df)

# 4. Define final matrix
predictor_columns = [
    'home_has_advantage', 'match_importance',
    'home_days_rest', 'home_win_rate', 'home_avg_goals_scored', 'home_avg_goals_conceded',
    'away_days_rest', 'away_win_rate', 'away_avg_goals_scored', 'away_avg_goals_conceded',
    'h2h_total_matches', 'h2h_home_win_percentage', 'h2h_goal_diff_historical'
]

X = model_ready_df[predictor_columns]
y = model_ready_df['outcome']

# Display results
print(f"Pipeline executed successfully!")
print(f"Feature Matrix Shape: {X.shape}")
X.head()

Pipeline executed successfully!
Feature Matrix Shape: (49819, 13)


,home_has_advantage,match_importance,home_days_rest,home_win_rate,home_avg_goals_scored,home_avg_goals_conceded,away_days_rest,away_win_rate,away_avg_goals_scored,away_avg_goals_conceded,h2h_total_matches,h2h_home_win_percentage,h2h_goal_diff_historical
0,1,1.0,30.0,0.333333,0.50,0.50,30.0,0.750000,2.250000,0.750000,0,0.500000,0.0
1,1,1.0,30.0,0.750000,2.00,0.50,30.0,0.333333,0.500000,0.500000,1,0.000000,0.0
2,1,1.0,30.0,0.333333,1.00,2.00,30.0,0.750000,2.500000,0.750000,2,0.500000,2.0
3,1,1.0,30.0,0.500000,1.75,1.00,30.0,0.333333,1.333333,1.666667,3,0.666667,3.0
4,1,1.0,30.0,0.250000,1.50,1.75,30.0,0.250000,1.750000,1.500000,4,0.500000,3.0
